In [1]:
import pandas as pd

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet"
columns = ['PULocationID', 'DOLocationID', 'trip_distance','tip_amount', 'total_amount', 'lpep_pickup_datetime','lpep_dropoff_datetime','passenger_count']
df = pd.read_parquet(url, columns=columns)
df.head()

,PULocationID,DOLocationID,trip_distance,tip_amount,total_amount,lpep_pickup_datetime,lpep_dropoff_datetime,passenger_count
0,247,69,0.70,1.70,10.00,2025-10-01 00:21:47,2025-10-01 00:24:37,1.0
1,66,25,1.61,2.78,16.68,2025-10-01 00:14:03,2025-10-01 00:24:14,1.0
2,244,244,0.00,2.20,13.20,2025-10-01 00:16:44,2025-10-01 00:16:47,1.0
3,95,170,10.37,11.31,67.85,2025-10-01 00:07:36,2025-10-01 00:32:14,1.0
4,82,138,4.07,6.82,34.12,2025-09-30 21:10:29,2025-09-30 21:22:30,1.0


In [ ]:
df[]

In [4]:
from dataclasses import dataclass

@dataclass
class Ride:
    PULocationID: int
    DOLocationID: int
    trip_distance: float
    total_amount: float
    tip_amount: float
    lpep_pickup_datetime: int  # epoch milliseconds
    lpep_dropoff_datetime: int
    passenger_count: int

In [13]:
import dataclasses
def ride_serializer(ride):
    ride_dict = dataclasses.asdict(ride)
    json_str = json.dumps(ride_dict)
    return json_str.encode('utf-8')

In [14]:
def ride_from_row(row):
    return Ride(
        PULocationID=int(row['PULocationID']),
        DOLocationID=int(row['DOLocationID']),
        trip_distance=float(row['trip_distance']),
        total_amount=float(row['total_amount']),
        tip_amount=float(row['tip_amount']),
        lpep_pickup_datetime=int(row['lpep_pickup_datetime'].timestamp() * 1000),
        lpep_dropoff_datetime=int(row['lpep_dropoff_datetime'].timestamp() * 1000),
        passenger_count=int(row['passenger_count']),
    )

In [15]:
import json
from kafka import KafkaProducer

def json_serializer(data):
    return json.dumps(data).encode('utf-8')

server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=ride_serializer
)

In [17]:
from time import time
topic_name = 'green-trips'
t0 = time()

for _, row in df.iterrows():
    ride = ride_from_row(row)
    producer.send(topic_name, value=ride)
    print(f"Sent: {ride}")
    # time.sleep(0.01)

producer.flush()

t1 = time()
print(f'took {(t1 - t0):.2f} seconds')

Sent: Ride(PULocationID=247, DOLocationID=69, trip_distance=0.7, total_amount=10.0, tip_amount=1.7, lpep_pickup_datetime=1759278107000, lpep_dropoff_datetime=1759278277000, passenger_count=1)
Sent: Ride(PULocationID=66, DOLocationID=25, trip_distance=1.61, total_amount=16.68, tip_amount=2.78, lpep_pickup_datetime=1759277643000, lpep_dropoff_datetime=1759278254000, passenger_count=1)
Sent: Ride(PULocationID=244, DOLocationID=244, trip_distance=0.0, total_amount=13.2, tip_amount=2.2, lpep_pickup_datetime=1759277804000, lpep_dropoff_datetime=1759277807000, passenger_count=1)
Sent: Ride(PULocationID=95, DOLocationID=170, trip_distance=10.37, total_amount=67.85, tip_amount=11.31, lpep_pickup_datetime=1759277256000, lpep_dropoff_datetime=1759278734000, passenger_count=1)
Sent: Ride(PULocationID=82, DOLocationID=138, trip_distance=4.07, total_amount=34.12, tip_amount=6.82, lpep_pickup_datetime=1759266629000, lpep_dropoff_datetime=1759267350000, passenger_count=1)
Sent: Ride(PULocationID=129, 

ValueError: cannot convert float NaN to integer